
### Cell 1: Setup & Dependencies

In [ ]:
# Install required packages for NIfTI processing and deep learning
!pip install -q nibabel scipy torch tqdm --no-deps

import glob
import os
import shutil
import tarfile
import numpy as np
import nibabel as nib
import scipy.ndimage as ndimage
from tqdm import tqdm

print("Environment setup complete. nibabel version:", nib.__version__)

---

### Cell 2: Core Processing Functions

*We define the logic here first so it is ready for the batch loop.*

In [ ]:
def preprocess_mri_volume(volume_data, lower_percentile=1.0, upper_percentile=99.0):
    """
    Winsorizes extreme brightness values and performs Z-score normalization
    restricted strictly to non-zero brain tissue.
    """
    brain_mask = volume_data > 0
    if not brain_mask.any():
        return volume_data.astype(np.float32)

    brain_voxels = volume_data[brain_mask]
    low_val, high_val = np.percentile(brain_voxels, [lower_percentile, upper_percentile])
    clipped_data = np.clip(volume_data, low_val, high_val)

    mean_val = np.mean(clipped_data[brain_mask])
    std_val = np.std(clipped_data[brain_mask])
    if std_val < 1e-8:
        std_val = 1.0

    normalized_data = np.zeros_like(volume_data, dtype=np.float32)
    normalized_data[brain_mask] = (clipped_data[brain_mask] - mean_val) / std_val

    return normalized_data

print("Preprocessing functions loaded.")

---

### Cell 3: The Batch Processing & Archiving Pipeline

*This is the main engine. It loops through one `.tar` file at a time, processes it into a separate folder, appends it to your final dataset, and cleans up the disk before moving to the next.*

In [ ]:
# 1. Define Paths
brats_root = "/kaggle/input/datasets/dschettler8845/brats-2021-task1"
raw_temp_dir = "/kaggle/working/raw_temp"
processed_temp_dir = "/kaggle/working/processed_temp"
final_archive_path = "/kaggle/working/brats2021_processed.tar"

# Remove existing archive if you are re-running the cell
if os.path.exists(final_archive_path):
    os.remove(final_archive_path)

# Known outliers to drop
invalid_cases = {"BraTS2021_00495", "BraTS2021_00621", "BraTS2021_01163"}

tar_files = glob.glob(f"{brats_root}/*.tar")
print(f"Found {len(tar_files)} archives. Starting batch processing...")

# 2. Process Batch by Batch
for tar_path in tar_files:
    tar_name = os.path.basename(tar_path)
    print(f"\n--- Processing {tar_name} ---")

    # Step A: Extract to a temporary raw folder
    print("Extracting raw files...")
    os.makedirs(raw_temp_dir, exist_ok=True)
    with tarfile.open(tar_path, "r") as tf:
        tf.extractall(path=raw_temp_dir)

    # Step B: Process files into a separate processed folder
    os.makedirs(processed_temp_dir, exist_ok=True)
    patient_folders = [f for f in glob.glob(os.path.join(raw_temp_dir, "BraTS2021_*")) if os.path.isdir(f)]

    print("Normalizing MRI volumes...")
    for folder in tqdm(patient_folders, desc=tar_name):
        patient_id = os.path.basename(folder)

        # Filter outliers instantly
        if patient_id in invalid_cases:
            continue

        out_folder = os.path.join(processed_temp_dir, patient_id)
        os.makedirs(out_folder, exist_ok=True)

        for file_name in os.listdir(folder):
            in_file = os.path.join(folder, file_name)
            out_file = os.path.join(out_folder, file_name)

            if file_name.endswith(".nii.gz") and not file_name.endswith("_seg.nii.gz"):
                # Normalize and save MRI scans
                nii_img = nib.load(in_file)
                volume_data = nii_img.get_fdata()
                processed_data = preprocess_mri_volume(volume_data)

                processed_img = nib.Nifti1Image(processed_data, nii_img.affine, nii_img.header)
                nib.save(processed_img, out_file)
            elif file_name.endswith("_seg.nii.gz"):
                # Directly copy segmentation masks without altering them
                shutil.copy2(in_file, out_file)

    # Step C: Delete RAW uncompressed data to free up disk space immediately
    print("Cleaning up raw data...")
    shutil.rmtree(raw_temp_dir)

    # Step D: Append processed data to the final archive
    print("Appending processed data to final archive...")
    with tarfile.open(final_archive_path, "a") as archive:
        # arcname="brats2021" ensures the files inside the tar extract to a nice root folder later
        archive.add(processed_temp_dir, arcname="brats2021")

    # Step E: Delete PROCESSED uncompressed data
    print("Cleaning up processed data...")
    shutil.rmtree(processed_temp_dir)

print("\n*** ALL BATCHES COMPLETE! ***")

---

### Cell 4: Validation

*Run this to verify your new dataset exists and check its final size.*

In [ ]:
if os.path.exists(final_archive_path):
    final_size_gb = os.path.getsize(final_archive_path) / 1e9
    print(f"Success! Processed archive created at: {final_archive_path}")
    print(f"Final compressed size: {final_size_gb:.2f} GB")
else:
    print("Error: Archive not found.")